# Task 2: Medical Report Generation using Visual Language Model

**Postdoctoral Technical Challenge - Alfaisal University**

Use MedGemma (or alternative VLM) to generate natural language reports from chest X-ray images.

**Before you start:** Upload to Colab (e.g. into `/content/`):
1. **report_generation.py** (this task's script)
2. **Your Task 1 trained model** — e.g. `pneumonia_net_pneumonia.pth` or `resnet34_pneumonia.pth`

**Prerequisites:** Hugging Face account, accept MedGemma terms: https://huggingface.co/google/medgemma-4b-it

## 1. Setup

In [ ]:
!pip install -q transformers>=4.50 accelerate torch torchvision medmnist Pillow huggingface_hub

In [ ]:
# Ensure GPU is enabled (Runtime → Change runtime type → T4 GPU)
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4"

In [ ]:
# STEP: Paste your Hugging Face token below (get it from https://huggingface.co/settings/tokens)
# First accept MedGemma terms: https://huggingface.co/google/medgemma-4b-it (click "Access repository")
HF_TOKEN = ""  # <-- Paste your token between the quotes, e.g. HF_TOKEN = "hf_AbCdEf123..."

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    login()  # Will prompt you to paste token in the widget

## 1.1 Upload your Task 1 trained model

Upload your trained model from Task 1 (e.g. **pneumonia_net_pneumonia.pth** or **resnet34_pneumonia.pth**) to `/content/`. Run the cell below and choose the file when prompted.

In [ ]:
# Upload your trained .pth file to /content/ (run this cell and select the file)
from google.colab import files
import shutil
from pathlib import Path

uploaded = files.upload()
for name in uploaded:
    dest = Path("/content") / name
    with open(dest, "wb") as f:
        f.write(uploaded[name])
    print(f"Saved: {dest}")
if not uploaded:
    print("No file uploaded. You can also drag-and-drop pneumonia_net_pneumonia.pth into the Colab file browser (left sidebar → Files → upload to /content/).")

## 2. Run Report Generation Pipeline

In [ ]:
# Run pipeline (ensure report_generation.py and your .pth are in /content/)
# Option A: If you set HF_TOKEN in the login cell above:
!cd /content && python report_generation.py --n-samples 12

# Option B: Or pass token on command line (replace with your token):
# !cd /content && python report_generation.py --n-samples 12 --token "hf_YourTokenHere"

!find /content -name "generated_reports.json" 2>/dev/null

## 3. View Generated Reports

In [ ]:
import json
from pathlib import Path
import os

# Search for the file (Colab cwd is /content)
paths = [
    Path("/content/task2_outputs/generated_reports.json"),
    Path("task2_outputs/generated_reports.json"),
    Path(os.getcwd()) / "task2_outputs" / "generated_reports.json",
]
p = next((x for x in paths if x.exists()), None)
if not p:
    print("File not found. Run pipeline first. Search result:")
    os.system("find /content -name 'generated_reports.json' 2>/dev/null || echo 'Not found'")
else:
    with open(p) as f:
        reports = json.load(f)
    for r in reports[:5]:
        print(f"--- Sample {r['idx']} | GT: {r['ground_truth']} | CNN: {r['cnn_pred']} ({r['cnn_prob']:.2f}) | Correct: {r['cnn_correct']} ---")
        for strat, text in r["reports"].items():
            t = str(text)
            print(f"[{strat}]: {t[:250]}..." if len(t) > 250 else f"[{strat}]: {t}")
        print()